# 05 Evaluate WLASL1000 BiGRU + Temporal Attention Model

## Purpose
This notebook evaluates the trained WLASL1000 model and saves:

- overall metrics
- prediction-level results
- per-class performance
- common confusions
- confidence threshold analysis

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
PROJECT_ROOT = Path("E:/Be_My_Ear")
DATASET_NAME = "WLASL1000"
PREFIX = "wlasl1000"

BASE_DIR = PROJECT_ROOT / "data" / "processed" / "ASL" / DATASET_NAME
CLEAN_INDEX_FILE = BASE_DIR / f"{PREFIX}_clean_keypoint_index.csv"
LABEL_MAP_FILE = PROJECT_ROOT / "data" / "label_maps" / DATASET_NAME / f"asl_{PREFIX}_labels.json"

MODEL_DIR = PROJECT_ROOT / "models" / "ASL" / DATASET_NAME
MODEL_PATH = MODEL_DIR / f"bigru_attention_{PREFIX}.pt"
NORM_STATS_PATH = MODEL_DIR / f"{PREFIX}_train_norm_stats.npz"

REPORT_DIR = PROJECT_ROOT / "reports" / f"phase1_{PREFIX}"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Clean index:", CLEAN_INDEX_FILE.exists())
print("Label map:", LABEL_MAP_FILE.exists())
print("Model:", MODEL_PATH.exists())
print("Norm stats:", NORM_STATS_PATH.exists())

In [ ]:
df = pd.read_csv(CLEAN_INDEX_FILE)

with open(LABEL_MAP_FILE, "r", encoding="utf-8") as f:
    label_map = json.load(f)

id_to_gloss = {int(k): v["gloss"] for k, v in label_map.items()}
NUM_CLASSES = df["label_id"].nunique()

test_records = []
for label_id, group in df.groupby("label_id"):
    group = group.sample(frac=1, random_state=SEED).reset_index(drop=True)
    n = len(group)
    n_test = max(1, int(round(n * 0.15)))
    test_records.append(group.iloc[:n_test])

test_df = pd.concat(test_records).sample(frac=1, random_state=SEED).reset_index(drop=True)

norm_stats = np.load(NORM_STATS_PATH)
train_mean = norm_stats["mean"].astype(np.float32)
train_std = norm_stats["std"].astype(np.float32)

print("Clean samples:", len(df))
print("Test samples:", len(test_df))
print("Classes:", NUM_CLASSES)

In [ ]:
class SignKeypointDataset(Dataset):
    def __init__(self, dataframe, mean, std, use_velocity=True):
        self.dataframe = dataframe.reset_index(drop=True)
        self.mean = mean.reshape(1, -1).astype(np.float32)
        self.std = std.reshape(1, -1).astype(np.float32)
        self.use_velocity = use_velocity

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        keypoints = np.load(row["keypoint_path"]).astype(np.float32)
        keypoints = (keypoints - self.mean) / (self.std + 1e-6)

        velocity = np.zeros_like(keypoints, dtype=np.float32)
        velocity[1:] = keypoints[1:] - keypoints[:-1]
        features = np.concatenate([keypoints, velocity], axis=1)

        return torch.tensor(features, dtype=torch.float32), torch.tensor(int(row["label_id"]), dtype=torch.long)

test_loader = DataLoader(SignKeypointDataset(test_df, train_mean, train_std), batch_size=32, shuffle=False, num_workers=0)

In [ ]:
class BiGRUAttentionModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, num_layers=2, dropout=0.4):
        super().__init__()
        self.input_projection = nn.Sequential(nn.Linear(input_size, hidden_size), nn.LayerNorm(hidden_size), nn.ReLU(), nn.Dropout(dropout))
        self.gru = nn.GRU(hidden_size, hidden_size, num_layers=num_layers, batch_first=True, bidirectional=True, dropout=dropout if num_layers > 1 else 0)
        self.attention = nn.Sequential(nn.Linear(hidden_size * 2, hidden_size), nn.Tanh(), nn.Linear(hidden_size, 1))
        self.classifier = nn.Sequential(nn.Linear(hidden_size * 2, hidden_size), nn.LayerNorm(hidden_size), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_size, num_classes))

    def forward(self, x):
        x = self.input_projection(x)
        gru_out, _ = self.gru(x)
        scores = self.attention(gru_out).squeeze(-1)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        context = torch.sum(gru_out * weights, dim=1)
        return self.classifier(context)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(MODEL_PATH, map_location=device)

model = BiGRUAttentionModel(
    checkpoint.get("input_size", 516),
    256,
    checkpoint.get("num_classes", NUM_CLASSES),
    num_layers=2,
    dropout=0.4
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded checkpoint epoch:", checkpoint.get("epoch"))
print("Best validation F1:", checkpoint.get("best_val_f1"))

In [ ]:
all_labels, all_preds, all_probs = [], [], []

with torch.no_grad():
    for x, y in tqdm(test_loader, desc="Collecting predictions"):
        x = x.to(device)
        outputs = model(x)
        probs = F.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_labels.extend(y.numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

y_true = np.array(all_labels)
y_pred = np.array(all_preds)
y_probs = np.array(all_probs)

def top_k(y_true, y_probs, k):
    correct = 0
    for true_label, prob in zip(y_true, y_probs):
        if true_label in np.argsort(prob)[-k:]:
            correct += 1
    return correct / len(y_true)

test_top1 = accuracy_score(y_true, y_pred)
test_top3 = top_k(y_true, y_probs, 3)
test_top5 = top_k(y_true, y_probs, 5)
test_macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

print("=" * 80)
print("WLASL1000 Evaluation")
print("=" * 80)
print(f"Test Top-1 Accuracy: {test_top1:.4f}")
print(f"Test Top-3 Accuracy: {test_top3:.4f}")
print(f"Test Top-5 Accuracy: {test_top5:.4f}")
print(f"Test Macro F1: {test_macro_f1:.4f}")

In [ ]:
overall = pd.DataFrame([{
    "dataset": DATASET_NAME,
    "model": "BiGRU + Temporal Attention",
    "clean_samples": len(df),
    "classes": NUM_CLASSES,
    "test_samples": len(test_df),
    "test_top1_accuracy": test_top1,
    "test_top3_accuracy": test_top3,
    "test_top5_accuracy": test_top5,
    "test_macro_f1": test_macro_f1,
    "best_val_f1": checkpoint.get("best_val_f1"),
    "best_val_top5": checkpoint.get("best_val_top5"),
    "checkpoint_epoch": checkpoint.get("epoch")
}])

overall_file = REPORT_DIR / f"{PREFIX}_overall_metrics.csv"
overall.to_csv(overall_file, index=False)

print("Saved overall metrics:", overall_file)
overall

In [ ]:
prediction_records = []
test_df_reset = test_df.reset_index(drop=True)

for i in range(len(y_true)):
    true_id = int(y_true[i])
    pred_id = int(y_pred[i])
    confidence = float(y_probs[i][pred_id])
    top5_ids = np.argsort(y_probs[i])[-5:][::-1]

    prediction_records.append({
        "video_id": test_df_reset.iloc[i]["video_id"],
        "true_label_id": true_id,
        "true_gloss": id_to_gloss.get(true_id, str(true_id)),
        "predicted_label_id": pred_id,
        "predicted_gloss": id_to_gloss.get(pred_id, str(pred_id)),
        "confidence": confidence,
        "correct_top1": true_id == pred_id,
        "correct_top5": true_id in top5_ids,
        "top5_glosses": ", ".join([id_to_gloss.get(int(x), str(x)) for x in top5_ids])
    })

predictions_df = pd.DataFrame(prediction_records)
predictions_file = REPORT_DIR / f"{PREFIX}_test_predictions.csv"
predictions_df.to_csv(predictions_file, index=False)

print("Saved predictions:", predictions_file)
predictions_df.head()

In [ ]:
per_class_records = []

for label_id in sorted(np.unique(y_true)):
    mask = y_true == label_id
    per_class_records.append({
        "label_id": int(label_id),
        "gloss": id_to_gloss.get(int(label_id), str(label_id)),
        "test_samples": int(mask.sum()),
        "top1_accuracy": accuracy_score(y_true[mask], y_pred[mask]),
        "top3_accuracy": top_k(y_true[mask], y_probs[mask], 3),
        "top5_accuracy": top_k(y_true[mask], y_probs[mask], 5)
    })

per_class_df = pd.DataFrame(per_class_records)
per_class_file = REPORT_DIR / f"{PREFIX}_per_class_performance.csv"
per_class_df.to_csv(per_class_file, index=False)

print("Saved per-class performance:", per_class_file)
per_class_df.sort_values("top1_accuracy", ascending=False).head(20)

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))

confusion_records = []
for true_label in range(NUM_CLASSES):
    for pred_label in range(NUM_CLASSES):
        count = cm[true_label, pred_label]
        if true_label != pred_label and count > 0:
            confusion_records.append({
                "true_label_id": true_label,
                "true_gloss": id_to_gloss.get(true_label, str(true_label)),
                "predicted_label_id": pred_label,
                "predicted_gloss": id_to_gloss.get(pred_label, str(pred_label)),
                "count": int(count)
            })

confusion_df = pd.DataFrame(confusion_records)
if len(confusion_df) > 0:
    confusion_df = confusion_df.sort_values("count", ascending=False)

confusion_file = REPORT_DIR / f"{PREFIX}_common_confusions.csv"
confusion_df.to_csv(confusion_file, index=False)

print("Saved common confusions:", confusion_file)
confusion_df.head(20)

In [ ]:
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

threshold_records = []

for threshold in thresholds:
    confident = predictions_df[predictions_df["confidence"] >= threshold]
    threshold_records.append({
        "confidence_threshold": threshold,
        "coverage": len(confident) / len(predictions_df),
        "top1_accuracy_on_confident_samples": confident["correct_top1"].mean() if len(confident) else np.nan,
        "top5_accuracy_on_confident_samples": confident["correct_top5"].mean() if len(confident) else np.nan,
        "num_confident_samples": len(confident)
    })

threshold_df = pd.DataFrame(threshold_records)
threshold_file = REPORT_DIR / f"{PREFIX}_confidence_threshold_analysis.csv"
threshold_df.to_csv(threshold_file, index=False)

print("Saved confidence threshold analysis:", threshold_file)
threshold_df